In [3]:
!pip install yfinance

  Using cached yfinance-0.2.66-py2.py3-none-any.whl.metadata (6.0 kB)
  Using cached multitasking-0.0.12-py3-none-any.whl
  Using cached frozendict-2.4.7-cp310-cp310-win_amd64.whl.metadata (24 kB)
  Using cached peewee-3.18.3-py3-none-any.whl
  Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl.metadata (13 kB)
  Using cached websockets-15.0.1-cp310-cp310-win_amd64.whl.metadata (7.0 kB)
  Using cached soupsieve-2.8-py3-none-any.whl.metadata (4.6 kB)
  Using cached cffi-2.0.0-cp310-cp310-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-2.23-py3-none-any.whl.metadata (993 bytes)
Using cached yfinance-0.2.66-py2.py3-none-any.whl (123 kB)
Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl (1.6 MB)
Using cached cffi-2.0.0-cp310-cp310-win_amd64.whl (182 kB)
Using cached frozendict-2.4.7-cp310-cp310-win_amd64.whl (37 kB)
Using cached soupsieve-2.8-py3-none-any.whl (36 kB)
Using cached websockets-15.0.1-cp310-cp310-win_amd64.whl (176 kB)
Using cached pycparser-2.23-py3-none-any.w

In [4]:
!pip install langchain-google-genai

  Using cached langchain_google_genai-4.0.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached google_genai-1.55.0-py3-none-any.whl.metadata (47 kB)
  Using cached langchain_core-1.1.3-py3-none-any.whl.metadata (3.7 kB)
  Using cached anyio-4.12.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-win_amd64.whl.metadata (2.4 kB)
  Using cached uuid_utils-0.12.0-cp39-abi3-win_amd64.whl.metadata (1.1 kB)
  Using cached jsonpointer-3.0.0-py2.py3-none-any.whl.metadata (2.3 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3

In [5]:
from dotenv import load_dotenv
import os

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
sender_email = os.getenv("SENDER_EMAIL")
sender_password = os.getenv("SENDER_PASSWORD")


In [9]:
# --- Investment Agent with Groq + LangGraph 

from dotenv import load_dotenv
import os
import re
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

import yfinance as yf
from langgraph.graph import StateGraph
from langchain_groq import ChatGroq

from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
sender_email = os.getenv("SENDER_EMAIL")
sender_password = os.getenv("SENDER_PASSWORD")

from langchain_groq import ChatGroq

  
# ================
# Initialize LLM
# ================
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=GROQ_API_KEY,
    temperature=0.3,
    max_tokens=512,
)

# ==========================
# ETF whitelist & helpers
# ==========================
ETF_WHITELIST = {
    "stocks": {"VTI", "VOO", "ITOT", "SCHB"},
    "bonds":  {"BND", "AGG", "SCHZ"},
    "cash":   {"SGOV", "SHV", "BIL", "ICSH"},  # cash-like, ultra-short treasuries
}
ALL_TICKERS = set().union(*ETF_WHITELIST.values())
BLOCK_WORDS = {"ETF", "ETFs", "US", "USA", "CASH", "BONDS", "STOCKS"}

def extract_whitelisted_tickers(text: str):
    """Extract plausible tickers from plain text and filter to whitelist."""
    # Find tokens like VTI, BND, SGOV (2–5 uppercase letters)
    raw = re.findall(r"\b[A-Z]{2,5}\b", text)
    raw = [t for t in raw if t not in BLOCK_WORDS]
    whitelisted = [t for t in raw if t in ALL_TICKERS]
    # De-dup while preserving order
    seen, out = set(), []
    for t in whitelisted:
        if t not in seen:
            seen.add(t)
            out.append(t)
    return out

# ==========================
# LangGraph nodes
# ==========================

# 1) User Input
def get_user_input(state):
    state["user_input"] = {
        "age": 30,
        "goal": "retirement",
        "horizon": 25,
        "income": 70000,
        "email": "bhumikasanap845@gmail.com",
    }
    return state

# 2) Profile Analyzer
def profile_analyzer(state):
    profile = state["user_input"]
    prompt = f"""
Analyze this user profile:
- Age: {profile['age']}
- Goal: {profile['goal']}
- Investment Horizon: {profile['horizon']} years
- Income: ${profile['income']}

Give a short summary and note any key financial constraints.
"""
    response = llm.invoke(prompt)
    state["profile_summary"] = response.content
    return state

# 3) Risk Model
def risk_model(state):
    profile = state["user_input"]
    prompt = f"""
Based on the user's age ({profile['age']}) and investment horizon ({profile['horizon']}), 
estimate their risk tolerance (low, medium, high) with a 1-line justification.
"""
    response = llm.invoke(prompt)
    state["risk_profile"] = response.content
    return state

# 4) Asset Allocator
def asset_allocator(state):
    prompt = f"""
Based on this risk profile: {state['risk_profile']}, suggest an asset allocation.
Return exactly one line in this strict format (no extra words):
Stocks % | Bonds % | Cash %
"""
    response = llm.invoke(prompt)
    # Light normalization (optional): strip spaces/newlines
    state["allocation"] = response.content.strip()
    return state

# 5) Investment Suggestion (plain text; no JSON)
def investment_suggestion(state):
    prompt = f"""
You are a financial assistant. Based on this allocation: {state['allocation']},
recommend exactly ONE ETF for each category using ONLY these tickers:

Stocks: {", ".join(sorted(ETF_WHITELIST['stocks']))}
Bonds:  {", ".join(sorted(ETF_WHITELIST['bonds']))}
Cash:   {", ".join(sorted(ETF_WHITELIST['cash']))}

Return PLAIN TEXT in exactly three lines, no extra text:
Stocks: TICKER - one short reason
Bonds:  TICKER - one short reason
Cash:   TICKER - one short reason
"""
    response = llm.invoke(prompt)
    state["etf_suggestions"] = response.content.strip()
    return state

# 6) Fetch Live Prices
def fetch_etf_prices(state):
    text = state["etf_suggestions"]
    found = extract_whitelisted_tickers(text)

    # Pick one from each bucket; fallback to defaults if missing
    picks = {"stocks": None, "bonds": None, "cash": None}
    for t in found:
        if t in ETF_WHITELIST["stocks"] and not picks["stocks"]:
            picks["stocks"] = t
        elif t in ETF_WHITELIST["bonds"] and not picks["bonds"]:
            picks["bonds"] = t
        elif t in ETF_WHITELIST["cash"] and not picks["cash"]:
            picks["cash"] = t

    if not picks["stocks"]: picks["stocks"] = "VTI"
    if not picks["bonds"]:  picks["bonds"]  = "BND"
    if not picks["cash"]:   picks["cash"]   = "SGOV"

    tickers = [picks["stocks"], picks["bonds"], picks["cash"]]

    etf_data = {}
    for ticker in tickers:
        try:
            hist = yf.Ticker(ticker).history(period="5d")["Close"]
            etf_data[ticker] = {str(d.date()): round(float(p), 2) for d, p in hist.items()}
        except Exception as e:
            etf_data[ticker] = f"Error fetching: {e}"

    state["picked_tickers"] = picks
    state["etf_prices"] = etf_data
    return state

# 7) Final Summary
def final_summary(state):
    picks = state.get("picked_tickers", {})
    injected = state["etf_suggestions"]
    if picks:
        injected += (
            f"\n\nFinal chosen tickers:\n"
            f"Stocks: {picks['stocks']}\n"
            f"Bonds:  {picks['bonds']}\n"
            f"Cash:   {picks['cash']}\n"
        )

    summary = f"""
    Profile Summary:
{state['profile_summary']}

    Risk Profile:
{state['risk_profile']}

    Allocation Plan:
{state['allocation']}
    
    Recommended ETFs:
{injected}

    Live ETF Prices (Last 5 Days):
"""
    for ticker, prices in state["etf_prices"].items():
        summary += f"\n{ticker}:\n"
        if isinstance(prices, dict):
            for date, price in prices.items():
                summary += f"  {date}: ${price}\n"
        else:
            summary += f"  {prices}\n"

    state["summary"] = summary
    return state

# 8) Email Notification (Gmail + App Password required)
def send_email_notification(state):
    user_email = state["user_input"]["email"]
    summary = state["summary"]

    msg = MIMEMultipart()
    msg["From"] = sender_email
    msg["To"] = user_email
    msg["Subject"] = "📊 Your Personalized Investment Report"
    msg.attach(MIMEText(summary, "plain"))

    try:
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=30) as server:
            server.ehlo()
            server.starttls()
            server.ehlo()
            server.login(sender_email, sender_password)  # Must be a Gmail App Password
            server.send_message(msg)
        print(f"✅ Email sent successfully to {user_email}")
    except smtplib.SMTPAuthenticationError as e:
        print("❌ Gmail auth failed. Use a Gmail *App Password* and ensure 'From' matches the Gmail account.")
        print(f"Server says: {e}")
    except Exception as e:
        print(f"❌ Email sending failed: {e}")

    return state

# ==========================
# Build & Run the Graph
# ==========================
graph = StateGraph(dict)
graph.add_node("input", get_user_input)
graph.add_node("analyzer", profile_analyzer)
graph.add_node("risk", risk_model)
graph.add_node("allocate", asset_allocator)
graph.add_node("suggest", investment_suggestion)
graph.add_node("prices", fetch_etf_prices)
graph.add_node("summary", final_summary)
graph.add_node("email", send_email_notification)

graph.set_entry_point("input")
graph.add_edge("input", "analyzer")
graph.add_edge("analyzer", "risk")
graph.add_edge("risk", "allocate")
graph.add_edge("allocate", "suggest")
graph.add_edge("suggest", "prices")
graph.add_edge("prices", "summary")
graph.add_edge("summary", "email")
graph.set_finish_point("email")

if __name__ == "__main__":
    result = graph.compile().invoke({})
    print(result["summary"])


✅ Email sent successfully to bhumikasanap845@gmail.com

    Profile Summary:
**Summary:**

The user is a 30-year-old individual with a goal of retirement in 25 years. They have a relatively high income of $70,000 and a long investment horizon, which is a positive indicator for long-term financial planning. Given their age and investment horizon, they have a good opportunity to take advantage of compound interest and potentially achieve their retirement goal.

**Key Financial Constraints:**

1. **High income, but high expenses**: With a high income, the user may be prone to overspending and accumulating debt, which could hinder their ability to save and invest for retirement.
2. **Limited time to recover from market downturns**: Although they have a long investment horizon, they are still relatively young and may not have the luxury of waiting out market downturns. This could lead to increased stress and anxiety about their investments.
3. **Potential for lifestyle inflation**: As their